In [1]:
!pip install requests beautifulsoup4 trafilatura

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.9/151.9 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.8/248.8 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.5/316.5 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 14.2 MB/s eta 0:00:00
  Attempting uninstall: lxml
    Found existing installation: lxml 6.1.0
    Uninstalling lxml-6.1.0:
      Successfully uninstalled lxml-6.1.0
  Attempting uninstall: charset_normalizer
    Found existing installation: charset-normalizer 3.4.7
    Uninstalling charset-normalizer-3.4.7:
      Successfully uninstalled charset-normalizer-3.4.7
ERROR: pip's dependency resolver does not currently take into account all the packages that are

In [2]:
import os
import json
import re
import unicodedata
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

import requests
from bs4 import BeautifulSoup
import trafilatura

# Configuration
NEPALI_SCRIPT_RANGE = ('\u0900', '\u097F')
CHARS_PER_TOKEN = 3.5
MAX_THREADS = 50

# Sitemaps for fast link discovery
SITEMAPS = [
    # Removed ratopati and onlinekhabar as they block/redirect sitemap requests
    "https://www.setopati.com/sitemap.xml", 
]

# HTML seeds with pagination style (url, style)
# style can be "query" (?page=N) or "path" (/page/N)
HTML_SEEDS = [
    ("https://www.onlinekhabar.com/content/news", "path"),
    ("https://ratopati.com/category/news", "query"),
    ("https://www.setopati.com/social", "query"),
    ("https://www.setopati.com/kinmel", "query"),
    ("https://www.setopati.com/opinion", "query"),
    ("https://www.setopati.com/sports", "query"),
]

# eKantipur categories for API based fast discovery
EKANTIPUR_CATEGORIES = [
    "https://ekantipur.com/news",
    "https://ekantipur.com/business",
    "https://ekantipur.com/opinion",
    "https://ekantipur.com/sports",
]

def normalize_text(text):
    """Normalize unicode and whitespace."""
    text = unicodedata.normalize('NFC', text)
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def get_script_ratio(text, script_range):
    """Check what fraction of characters are in the target script."""
    if not text: return 0.0
    script_chars = sum(1 for c in text if script_range[0] <= c <= script_range[1])
    alpha_chars = sum(1 for c in text if c.isalpha())
    return script_chars / max(alpha_chars, 1)

def get_links_from_sitemap(sitemap_url, session, visited, depth=0):
    """Recursively fetch article links from a sitemap (or sitemap index)."""
    if depth > 2: # Limit recursion to avoid infinite loops
        return set()
        
    links = set()
    try:
        print(f"  Fetching sitemap: {sitemap_url}")
        resp = session.get(sitemap_url, timeout=20)
        if resp.status_code == 200:
            # Using lxml-xml if available, else standard xml
            try:
                soup = BeautifulSoup(resp.content, 'lxml-xml')
            except Exception:
                soup = BeautifulSoup(resp.content, 'xml')
            
            # Check if it's a sitemap index (points to other sitemaps)
            sitemaps = soup.find_all('sitemap')
            if sitemaps:
                print(f"    Found index with {len(sitemaps)} sub-sitemaps.")
                for s in sitemaps:
                    loc = s.find('loc')
                    if loc and loc.text:
                        # Some sites have huge sitemap indexes, we might want to prioritize recent ones
                        # Usually they are sorted, e.g., sitemap-2023.xml, etc.
                        sub_links = get_links_from_sitemap(loc.text.strip(), session, visited, depth + 1)
                        links.update(sub_links)
            else:
                # It's a regular sitemap with URLs
                urls = soup.find_all('url')
                print(f"    Found {len(urls)} URLs.")
                for u in urls:
                    loc = u.find('loc')
                    if loc and loc.text:
                        link = loc.text.strip()
                        if link not in visited:
                            links.add(link)
    except requests.exceptions.Timeout:
         print(f"  Timeout fetching sitemap {sitemap_url}")
    except Exception as e:
        print(f"  Error fetching sitemap {sitemap_url}: {e}")
        
    return links

def get_links_from_html(seed_url, pagination_style, target_pages, session, visited):
    """Scrape article links from an HTML page across multiple pages."""
    links = set()
    from urllib.parse import urlparse
    base_domain = urlparse(seed_url).netloc
    
    for page in range(1, target_pages + 1):
        # Construct paginated URL
        if page == 1:
            url = seed_url
        elif pagination_style == "query":
            # e.g., ?page=2 or &page=2
            separator = "&" if "?" in seed_url else "?"
            url = f"{seed_url}{separator}page={page}"
        elif pagination_style == "path":
            # e.g., /page/2
            url = seed_url.rstrip("/") + f"/page/{page}"
        else:
            url = seed_url
            
        try:
            resp = session.get(url, timeout=15)
            if resp.status_code != 200:
                print(f"  Reached end of pagination or error at {url} (Status: {resp.status_code})")
                break
                
            soup = BeautifulSoup(resp.content, 'html.parser')
            page_links = 0
            
            for a in soup.find_all('a', href=True):
                href = a['href']
                if href.startswith('/'):
                    href = f"https://{base_domain}{href}"
                
                # Check if it's the same domain and not visited
                if base_domain in href and href not in visited:
                    if not any(pattern in href for pattern in ['/author/', '/tag/', '/category/', '/video/']):
                        links.add(href)
                        visited.add(href)
                        page_links += 1
                        
            print(f"  [Page {page}] {url} -> Found {page_links} links.")
            
            if page_links == 0:
                break
                
        except Exception as e:
            print(f"  Error fetching {url}: {e}")
            break
            
    return links

def get_ekantipur_links(category_url, target_count, session, visited):
    """Fetch article links from eKantipur using their internal API pagination."""
    cat_slug = category_url.rstrip('/').split('/')[-1]
    
    print(f"  Fetching ekantipur API for category: {cat_slug}")
    try:
        resp = session.get(category_url, timeout=15)
        if resp.status_code != 200:
            return set()
    except Exception as e:
        print(f"    Error accessing eKantipur category: {e}")
        return set()
    
    soup = BeautifulSoup(resp.text, 'html.parser')
    request_date_input = soup.find('input', {'id': 'hdnRequestDate'})
    last_news_date_input = soup.find('input', {'id': 'hdnLastNewsDate'})
    current_offset_input = soup.find('input', {'id': 'hdnCurrentOffset'})
    
    request_date = request_date_input['value'] if request_date_input else datetime.now().strftime('%Y/%m/%d')
    last_news_date = last_news_date_input['value'] if last_news_date_input else request_date
    offset = int(current_offset_input['value']) if current_offset_input else 30
    
    links = set()
    api_url = "https://api.ekantipur.com/kantipur/v4/web/category"
    api_headers = {
        'Content-Type': 'application/x-www-form-urlencoded',
        'Origin': 'https://ekantipur.com',
        'Referer': category_url
    }
    
    consecutive_failures = 0
    max_failures = 3
    
    while len(links) < target_count:
        post_data = {
            'json': 'true',
            'deviceSource': 'desktop',
            'currentslug': cat_slug,
            'offset': str(offset),
            'requestDate': request_date,
            'requestLastNewsDate': last_news_date,
            'enNews': '',
            'hdnLN': ''
        }
        
        try:
            resp = session.post(api_url, data=post_data, headers=api_headers, timeout=15)
            if resp.status_code != 200:
                consecutive_failures += 1
                if consecutive_failures >= max_failures:
                    break
                continue
            
            data = resp.json()
            consecutive_failures = 0
            
            html_content = data.get('html', '')
            if not html_content:
                break
                
            page_soup = BeautifulSoup(html_content, 'html.parser')
            new_links = 0
            for a_tag in page_soup.find_all('a', href=True):
                href = a_tag['href']
                if href.startswith('/'):
                    href = f"https://ekantipur.com{href}"
                if 'ekantipur.com' in href and f'/{cat_slug}/' in href.lower() and href not in visited:
                    if not any(pattern in href for pattern in ['/author/', '/tag/', '/page/']):
                        links.add(href)
                        new_links += 1
            
            next_offset = data.get('nextOffset')
            if next_offset:
                offset = next_offset
            else:
                break
                
            last_news_date = data.get('lastNewsDate') or last_news_date
            request_date = data.get('date') or data.get('lastDate') or request_date
            
            if new_links == 0:
                consecutive_failures += 1
                if consecutive_failures >= max_failures:
                    break
                    
            print(f"    -> {len(links):,} links (offset={offset})")
            
        except Exception:
            consecutive_failures += 1
            if consecutive_failures >= max_failures:
                break
                
    return links

def scrape_article(url, session):
    """Scrape the text of a single article using trafilatura."""
    try:
        resp = session.get(url, timeout=15)
        if resp.status_code != 200:
            return url, None
        
        text = trafilatura.extract(resp.content, include_comments=False, include_tables=False)
        return url, text
    except Exception:
        return url, None

def main():
    import argparse
    parser = argparse.ArgumentParser(description="Fast Nepali Crawler")
    parser.add_argument('--output_dir', type=str, default='./data', help='Output directory')
    parser.add_argument('--target_tokens', type=int, default=100_000_000, help='Target token count')
    parser.add_argument('--workers', type=int, default=MAX_THREADS, help='Number of worker threads')
    parser.add_argument('--ekantipur_target', type=int, default=20000, help='Ekantipur articles per category')
    parser.add_argument('--html_pages', type=int, default=300, help='Number of pages to paginate through for HTML seeds')
    args, unknown = parser.parse_known_args()
    
    output_dir = os.path.join(args.output_dir, 'nepali', 'fast_scrape')
    os.makedirs(output_dir, exist_ok=True)
    output_file = os.path.join(output_dir, 'scraped_nepali.jsonl')
    
    visited_urls = set()
    total_est_tokens = 0
    total_articles = 0
    
    if os.path.exists(output_file):
        print(f"Resuming from existing file: {output_file}")
        with open(output_file, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    record = json.loads(line)
                    visited_urls.add(record['source_name'])
                    char_count = record.get('clean_char_count', len(record['text']))
                    total_est_tokens += int(char_count / CHARS_PER_TOKEN)
                    total_articles += 1
                except json.JSONDecodeError:
                    pass
        print(f"Resumed state: {total_articles:,} articles, ~{total_est_tokens:,} tokens.\n")
        
    session = requests.Session()
    session.headers.update({'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/110.0.0.0 Safari/537.36'})
    # Connection pooling for high concurrency
    adapter = requests.adapters.HTTPAdapter(pool_connections=args.workers, pool_maxsize=args.workers, max_retries=2)
    session.mount('https://', adapter)
    session.mount('http://', adapter)
    
    print("=== PHASE 1: Fast Link Discovery ===")
    all_links = set()
    
    # 1. Gather links from Sitemaps
    if SITEMAPS:
        for sitemap in SITEMAPS:
            links = get_links_from_sitemap(sitemap, session, visited_urls)
            all_links.update(links)
            
    # 2. Gather links from HTML Seeds using pagination
    if HTML_SEEDS:
        for seed_url, style in HTML_SEEDS:
            print(f"\nScanning HTML Seed: {seed_url}")
            links = get_links_from_html(seed_url, style, args.html_pages, session, visited_urls)
            all_links.update(links)
        
    # 3. Gather links from eKantipur API
    for category in EKANTIPUR_CATEGORIES:
        links = get_ekantipur_links(category, args.ekantipur_target, session, visited_urls)
        all_links.update(links)
        
    all_links = list(all_links)
    print(f"\nPhase 1 Complete. Total unique NEW links discovered: {len(all_links):,}")
    
    if not all_links:
        print("No new links to scrape. Exiting.")
        return

    print("\n=== PHASE 2: High-Speed Scraping ===")
    with open(output_file, 'a', encoding='utf-8') as f:
        with ThreadPoolExecutor(max_workers=args.workers) as executor:
            # Submit links in chunks to manage memory
            chunk_size = 5000
            for i in range(0, len(all_links), chunk_size):
                if total_est_tokens >= args.target_tokens:
                    break
                    
                chunk = all_links[i:i+chunk_size]
                futures = {executor.submit(scrape_article, url, session): url for url in chunk}
                
                for future in as_completed(futures):
                    url, text = future.result()
                    
                    if text:
                        text = normalize_text(text)
                        
                        # Validate it is Nepali (Devanagari script)
                        if len(text) > 200 and get_script_ratio(text, NEPALI_SCRIPT_RANGE) > 0.5:
                            char_count = len(text)
                            est_tokens = int(char_count / CHARS_PER_TOKEN)
                            
                            record = {
                                'text': text,
                                'source_name': url,
                                'source_type': 'fast_scrape',
                                'doc_id': f'scrape_nepali_{total_articles}',
                                'clean_char_count': char_count,
                                'est_tokens': est_tokens
                            }
                            f.write(json.dumps(record, ensure_ascii=False) + '\n')
                            
                            total_articles += 1
                            total_est_tokens += est_tokens
                            
                            if total_articles % 100 == 0:
                                print(f"Progress: {total_articles:,} articles | ~{total_est_tokens:,} / {args.target_tokens:,} tokens")
                                
                    if total_est_tokens >= args.target_tokens:
                        print("\nTarget token count reached!")
                        break

    print("\n" + "="*50)
    print("Scraping Completed.")
    print(f"Total Articles: {total_articles:,}")
    print(f"Total Tokens:   ~{total_est_tokens:,}")
    print(f"Output saved to: {output_file}")
    print("="*50)

if __name__ == "__main__":
    main()


=== PHASE 1: Fast Link Discovery ===
  Fetching sitemap: https://www.setopati.com/sitemap.xml
    Found 1 URLs.

Scanning HTML Seed: https://www.onlinekhabar.com/content/news
  [Page 1] https://www.onlinekhabar.com/content/news -> Found 268 links.
  [Page 2] https://www.onlinekhabar.com/content/news/page/2 -> Found 193 links.
  [Page 3] https://www.onlinekhabar.com/content/news/page/3 -> Found 192 links.
  [Page 4] https://www.onlinekhabar.com/content/news/page/4 -> Found 194 links.
  [Page 5] https://www.onlinekhabar.com/content/news/page/5 -> Found 194 links.
  [Page 6] https://www.onlinekhabar.com/content/news/page/6 -> Found 194 links.
  [Page 7] https://www.onlinekhabar.com/content/news/page/7 -> Found 194 links.
  [Page 8] https://www.onlinekhabar.com/content/news/page/8 -> Found 194 links.
  [Page 9] https://www.onlinekhabar.com/content/news/page/9 -> Found 194 links.
  [Page 10] https://www.onlinekhabar.com/content/news/page/10 -> Found 194 links.
  [Page 11] https://www.online